## 1. Install Dependencies

In [ ]:
!pip install -q transformers==4.44.2 tokenizers datasets \
             sentencepiece rouge-score accelerate py7zr evaluate

## 2. Environment Check

In [ ]:
import torch
import platform
import psutil

print("=== Environment Info ===")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU count       : {torch.cuda.device_count()}")
    print(f"Current device  : {torch.cuda.current_device()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  - Total memory: {props.total_memory / 1e9:.2f} GB")
else:
    print("No GPU detected. Training will run on CPU.")

ram = psutil.virtual_memory()
print(f"RAM total       : {ram.total / 1e9:.2f} GB")
print(f"Device used     : {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Platform        : {platform.platform()}")

## 3. Paths & Config

In [ ]:
import os

DATASET_SLUG = "datasets/zanzungg/dataset"
DATA_DIR     = f"/kaggle/input/{DATASET_SLUG}"

OUTPUT_DIR   = "/kaggle/working/bartpho-finetuned"
LOG_DIR      = "/kaggle/working/logs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("=== Dataset Check ===")
print(f"DATA_DIR = {DATA_DIR}")

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")

print("Files in dataset directory:")
print(os.listdir(DATA_DIR))

for split in ["train", "val", "test"]:
    path = f"{DATA_DIR}/{split}.jsonl"
    if not os.path.exists(path):
        print(f"[ERROR] {split}.jsonl not found at {path}")
        continue
    with open(path, "r", encoding="utf-8") as f:
        num_lines = sum(1 for _ in f)
    print(f"[OK] {split}.jsonl — {num_lines} samples")

print(f"\nOutput directory : {OUTPUT_DIR}")
print(f"Log directory    : {LOG_DIR}")

## 4. Load Dataset

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                data.append(obj)
            except json.JSONDecodeError:
                print(f"[WARNING] JSON decode error at line {i} in {path}")
    return data

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_data   = load_jsonl(f"{DATA_DIR}/val.jsonl")
test_data  = load_jsonl(f"{DATA_DIR}/test.jsonl")

dataset = DatasetDict({
    "train":      Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test":       Dataset.from_list(test_data),
})

print("=== Dataset Overview ===")
print(dataset)
print("\nColumns:", dataset["train"].column_names)

required_keys = ["article", "summary"]
for key in required_keys:
    if key not in dataset["train"].column_names:
        raise ValueError(f"Missing required key: {key}")

sample = dataset["train"][0]
print("\n=== Sample ===")
print(f"Article length : {len(sample['article'])} chars")
print(f"Summary length : {len(sample['summary'])} chars")
print(f"Article preview: {sample['article'][:200]}...")
print(f"Summary preview: {sample['summary'][:100]}...")

def avg_len(data, key):
    return sum(len(x[key]) for x in data) / len(data)

print("\n=== Length Statistics ===")
print(f"Avg article length: {avg_len(train_data, 'article'):.1f}")
print(f"Avg summary length: {avg_len(train_data, 'summary'):.1f}")

## 5. Load BARTpho Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig
import torch

MODEL_NAME  = "vinai/bartpho-syllable-base"
ARTICLE_KEY = "article"
SUMMARY_KEY = "summary"
MAX_INPUT   = 1024
MAX_TARGET  = 128

# 1. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# 2. Cấu hình Generation
gen_config = GenerationConfig()
gen_config.max_length           = MAX_TARGET
gen_config.early_stopping       = True
gen_config.no_repeat_ngram_size = 3
gen_config.num_beams            = 4

gen_config.pad_token_id         = tokenizer.pad_token_id
gen_config.eos_token_id         = tokenizer.eos_token_id
gen_config.forced_bos_token_id  = tokenizer.bos_token_id
gen_config.decoder_start_token_id = tokenizer.bos_token_id

model.generation_config = gen_config

# 3. Cấu hình hỗ trợ Training
model.config.use_cache = False  # Bắt buộc False khi dùng gradient checkpointing

# 4. Kiểm tra và chuyển thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 5. Tính toán thông số mô hình
total_params     = model.num_parameters()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== Model Info ===")
print(f"Model name       : {MODEL_NAME}")
print(f"Total params     : {total_params/1e6:.1f}M")
print(f"Trainable params : {trainable_params/1e6:.1f}M")
print(f"Vocab size       : {tokenizer.vocab_size}")
print(f"Device           : {device}")

print("\n=== Generation Config (Validated) ===")
print(model.generation_config)

## 6. Tokenization

In [ ]:
def preprocess_bartpho(examples):
    inputs  = examples[ARTICLE_KEY]
    targets = examples[SUMMARY_KEY]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = dataset.map(
    preprocess_bartpho,
    batched=True,
    batch_size=512,
    num_proc=2,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing (BARTpho)",
)

print("\nTokenization completed!")
print(tokenized)

sample = tokenized["train"][0]
print("\n=== Tokenized Sample ===")
print(f"Input length : {len(sample['input_ids'])}")
print(f"Label length : {len(sample['labels'])}")

decoded_input = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
print(f"\nDecoded input preview:\n{decoded_input[:200]}...")

## 7. Metrics (ROUGE)

In [ ]:
import evaluate
import numpy as np

rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id).astype(np.int32)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id).astype(np.int32)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True, clean_up_tokenization_spaces=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True, clean_up_tokenization_spaces=True)

    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    return {k: round(v * 100, 4) for k, v in result.items()}

## 8. Training Arguments

In [ ]:
import math
from transformers import Seq2SeqTrainingArguments

num_gpus = max(1, torch.cuda.device_count())

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Epochs & Batch ────────────────────────────────────────────────
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,

    # ── Bộ nhớ ────────────────────────────────────────────────────────
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_safetensors=False,

    # ── Tối ưu hóa ────────────────────────────────────────────────────
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,

    # ── Đánh giá & Lưu trữ ────────────────────────────────────────────
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    # ── Generation (eval) ─────────────────────────────────────────────
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    generation_num_beams=4,

    # ── Khác ──────────────────────────────────────────────────────────
    logging_steps=50,
    report_to="none",
    group_by_length=True,
    dataloader_num_workers=4,
)

batch_eff       = training_args.per_device_train_batch_size * num_gpus * training_args.gradient_accumulation_steps
n_train         = len(tokenized["train"])
steps_per_epoch = math.ceil(n_train / batch_eff)
total_steps     = steps_per_epoch * training_args.num_train_epochs

print("=== Training Setup ===")
print(f"Train samples      : {n_train:,}")
print(f"Effective batch    : {batch_eff}")
print(f"Steps / epoch      : {steps_per_epoch:,}")
print(f"Total steps        : {total_steps:,}")
print(f"Warmup steps (~10%): {int(total_steps * training_args.warmup_ratio)}")

## 9. Trainer Setup & Training

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if training_args.fp16 else None,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print("=== Training Start ===")
print(f"Model        : {MODEL_NAME}")
print(f"Device       : {gpu_name}")
print(f"Epochs       : {training_args.num_train_epochs}")
print(f"FP16         : {training_args.fp16}")
print(f"Beam size    : {training_args.generation_num_beams}")
print(f"LR           : {training_args.learning_rate}")

print("\nStarting full training...")
trainer.train()

## 10. Evaluate on Test Set

In [ ]:
import numpy as np

print("Running evaluation on test set...")
test_results = trainer.predict(
    tokenized["test"],
    metric_key_prefix="test",
)
metrics = test_results.metrics

print("\n" + "="*50)
print("         TEST SET RESULTS")
print("="*50)
for k in ["test_rouge1", "test_rouge2", "test_rougeL", "test_rougeLsum"]:
    print(f"{k:20}: {metrics.get(k, 0):.2f}")
print("="*50)

preds  = test_results.predictions
labels = test_results.label_ids

preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

print("\n=== Sample Predictions ===")
for i in range(3):
    print(f"\n[Sample {i+1}]")
    original_article = dataset["test"][i]["article"]
    print(f"[-] Article (shortened): {original_article[:200]}...")
    print(f"[-] Ground Truth       : {decoded_labels[i]}")
    print(f"[-] Model Prediction   : {decoded_preds[i]}")
    print("-" * 30)

## 11. Save Model & Metrics

In [ ]:
import json

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(f"{OUTPUT_DIR}/test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Model saved to     : {OUTPUT_DIR}")
print(f"Metrics saved to   : {OUTPUT_DIR}/test_metrics.json")
print("\nFinal test metrics:")
print(json.dumps({k: v for k, v in metrics.items() if 'rouge' in k}, indent=2))

## 12. (Optional) Quick Inference Test

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

test_article = dataset["test"][0]["article"]

output = summarizer(
    test_article,
    truncation=True,
    max_length=MAX_TARGET,
    min_length=20,
    num_beams=4,
    no_repeat_ngram_size=4,
    early_stopping=True,
)

print("=== Inference Test ===")
print(f"Input   : {test_article[:300]}...")
print(f"Summary : {output[0]['summary_text']}")